**Навигация по уроку**

4. Домашняя работа

Используя модель обучения многослойного персептрона из практической части урока (9.3), выполните следующее:

1. Увеличьте число слоев до 4-х и сравните время обучения модели и точность на тестовой выборке.
2. В качестве датасета использовать любой из наборов данных TensorFlow. Обучить модель. Добейтесь результата распознования более 85% на тестовой выборке. [Датасеты на выбор](https://www.tensorflow.org/datasets/overview). Используйте датасет отличный от MNIST, который был в уроке.

Для прохождения урока достаточно решить первое задание.

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import time
import numpy as np

# Загружаем Fashion MNIST (датасет отличный от MNIST)
dataset, info = tfds.load('fashion_mnist', with_info=True, as_supervised=True)

# Информация о датасете
print(f"Датасет: {info.name}")
print(f"Количество классов: {info.features['label'].num_classes}")
print(f"Классы: {info.features['label'].names}\n")

# Разделяем на train и test
train_dataset = dataset['train']
test_dataset = dataset['test']

# Преобразуем в numpy массивы
def dataset_to_numpy(dataset):
    images = []
    labels = []
    for image, label in tfds.as_numpy(dataset):
        images.append(image)
        labels.append(label)
    return np.array(images), np.array(labels)

print("Загрузка данных...")
X_train, y_train = dataset_to_numpy(train_dataset)
X_test, y_test = dataset_to_numpy(test_dataset)

print(f"Размер тренировочных данных: {X_train.shape}")
print(f"Размер тестовых данных: {X_test.shape}\n")

# Нормализуем пиксели (0-255 -> 0-1)
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Для MLP нужно преобразовать 2D изображения в 1D векторы
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

print(f"После flatten: {X_train_flat.shape}")
print(f"Количество признаков на входе: {X_train_flat.shape[1]}")

# Создаем модели для сравнения
def create_model(num_layers=1):
    model = tf.keras.Sequential()

    # Входной слой
    model.add(tf.keras.layers.Input(shape=(X_train_flat.shape[1],)))

    # Скрытые слои
    if num_layers == 1:
        model.add(tf.keras.layers.Dense(128, activation='relu'))
    else:
        # Для 4 слоёв (3 скрытых)
        model.add(tf.keras.layers.Dense(256, activation='relu'))
        model.add(tf.keras.layers.Dense(128, activation='relu'))
        model.add(tf.keras.layers.Dense(64, activation='relu'))

    # Выходной слой (10 классов)
    model.add(tf.keras.layers.Dense(10, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Обучение модели с 1 слоем
print("=" * 50)
print("Обучение модели с 1 скрытым слоем")
print("=" * 50)

model_1 = create_model(num_layers=1)
start_1 = time.time()
history_1 = model_1.fit(
    X_train_flat, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)
end_1 = time.time()
time_1 = end_1 - start_1

# Оценка модели с 1 слоем
loss_1, acc_1 = model_1.evaluate(X_test_flat, y_test, verbose=0)

print("\n" + "=" * 50)
print("Обучение модели с 3 скрытыми слоями (всего 4 слоя)")
print("=" * 50)

# Обучение модели с 4 слоями
model_4 = create_model(num_layers=4)
start_4 = time.time()
history_4 = model_4.fit(
    X_train_flat, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)
end_4 = time.time()
time_4 = end_4 - start_4

# Оценка модели с 4 слоями
loss_4, acc_4 = model_4.evaluate(X_test_flat, y_test, verbose=0)

# Вывод результатов сравнения
print("\n" + "=" * 60)
print("Результаты сравнения")
print("=" * 60)

print(f"\n{'Модель':<25} {'Время':<15} {'Точность':<15}")
print("-" * 55)
print(f"{'1 скрытый слой':<25} {time_1:<15.2f} {acc_1:<15.4f}")
print(f"{'3 скрытых слоя (всего 4)':<25} {time_4:<15.2f} {acc_4:<15.4f}")

# Разница
time_diff = time_4 - time_1
acc_diff = acc_4 - acc_1

print("\n" + "=" * 60)
print("Выводы:")
print(f"• Время обучения увеличилось на {time_diff:.2f} сек ({time_diff/time_1*100:.1f}%)")
print(f"• Точность {'увеличилась' if acc_diff > 0 else 'уменьшилась'} на {abs(acc_diff):.4f} ({acc_diff*100:+.2f}%)")

if acc_diff > 0:
    print("✅ Больше слоёв помогло улучшить точность")
elif acc_diff < 0:
    print("⚠️ Больше слоёв ухудшило точность (возможно, переобучение)")
else:
    print("➡️ Точность не изменилась")

# Проверка условия >85%
if acc_4 > 0.85:
    print(f"\n✅ Условие выполнено: точность {acc_4:.2%} > 85%")
else:
    print(f"\n❌ Условие не выполнено: точность {acc_4:.2%} < 85%")